<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-06-advanced-rag-and-open-source-llms/notebooks/Ophelie_RAG_Configurations_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Run 3 RAG Configurations and Log Output Differences

**Objective:** Run the same question through three different retriever configurations (varying `top_k`, similarity threshold, and whether an LLM-based reranker is applied) and compare what gets retrieved and what the final answer looks like.

Document tested: `sample-sdf-document (Project 5).pdf` — the same Certificate of Quality / storage-letter document used throughout this project.

| Config | top_k | Threshold | Reranker |
|---|---|---|---|
| A (Default) | 5 | None | Off |
| B | 8 | 0.15 | Off |
| C | 5 | 0.15 | On (LLM-based) |


In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-google-genai pymupdf gdown nest_asyncio google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


## Step 1: Download the real reference document

In [ ]:
import gdown

FILE_ID = "1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN"
url = f"https://drive.google.com/uc?id={FILE_ID}"
output_path = "sample_sdf_document.pdf"

gdown.download(url, output_path, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN
To: /content/sample_sdf_document.pdf
100%|██████████| 147k/147k [00:00<00:00, 20.6MB/s]


'sample_sdf_document.pdf'

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(output_path)
full_text = ""
for page in doc:
    full_text += page.get_text()
num_pages = doc.page_count
doc.close()

print(f"Pages: {num_pages}")
print(f"Extracted characters: {len(full_text)}")
print(full_text[:500])


Pages: 3
Extracted characters: 4023
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could le


## Step 2: Chunk once and build a single index

Chunking and embedding are done **once** with the same settings used throughout this project (`chunk_size=200`, `chunk_overlap=30`, MiniLM embeddings), so the only thing that changes between configs A, B, and C is the retrieval logic (`top_k`, threshold, reranker) — not the underlying index.

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

document = Document(text=full_text)
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([document])
print(f"Number of chunks: {len(nodes)}")

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
index = VectorStoreIndex(nodes, embed_model=embed_model)
print("Vector index built successfully!")


Number of chunks: 6


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index built successfully!


## Step 3: Configure Gemini for answer generation

Same substitution used throughout this project: `llama_index.llms.gemini.Gemini` / `models/gemini-2.0-flash` returns 429 quota errors on the free tier. Using `llama_index.llms.google_genai.GoogleGenAI` with `models/gemini-flash-latest` instead.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI

api_key = userdata.get("GOOGLE_API_KEY")
llm = GoogleGenAI(model="models/gemini-flash-latest", api_key=api_key)

test_response = llm.complete("Reply with the single word: ready")
print("API key loaded:", "Yes" if api_key else "No")
print("Gemini test response:", test_response)


API key loaded: Yes
Gemini test response: ready


## Step 4: Define the query and the 3 retriever configurations

In [ ]:
query = "What test methods were used for quality control?"

configs = {
    "A": {"top_k": 5, "threshold": None, "reranker": False},   # Default baseline
    "B": {"top_k": 8, "threshold": 0.15, "reranker": False},   # More chunks, filter weak matches
    "C": {"top_k": 5, "threshold": 0.15, "reranker": True},    # Filtered + LLM-reranked
}

for label, cfg in configs.items():
    print(f"{label}: {cfg}")


A: {'top_k': 5, 'threshold': None, 'reranker': False}
B: {'top_k': 8, 'threshold': 0.15, 'reranker': False}
C: {'top_k': 5, 'threshold': 0.15, 'reranker': True}


## Step 5: A lightweight LLM-based reranker

LlamaIndex ships an `LLMRerank` postprocessor, but its output-parsing prompt is tuned for OpenAI-style models and can fail to parse cleanly with other LLMs. To keep this reliable, we use a small custom reranker: ask Gemini to look at the retrieved chunks and return their indices ordered from most to least relevant, then reorder the chunks accordingly. If parsing ever fails, we fall back to the original retrieval order rather than crashing.

In [ ]:
def llm_rerank(nodes, query, llm):
    """Ask the LLM to re-order retrieved chunks by relevance to the query."""
    listing = "\n".join(f"[{i}] {n.get_text()[:300]}" for i, n in enumerate(nodes))
    prompt = (
        "You are ranking document chunks by how relevant they are to a question.\n"
        f"Question: {query}\n\n"
        f"Chunks:\n{listing}\n\n"
        "Return ONLY a comma-separated list of the chunk indices, ordered from MOST relevant "
        "to LEAST relevant. Example: 2,0,1,3"
    )
    response = str(llm.complete(prompt)).strip()
    try:
        order = [int(x.strip()) for x in response.split(",") if x.strip().lstrip("-").isdigit()]
        order = [i for i in order if 0 <= i < len(nodes)]
        seen = set(order)
        order += [i for i in range(len(nodes)) if i not in seen]
        return [nodes[i] for i in order]
    except Exception as e:
        print(f"Rerank parsing failed ({e}); keeping original retrieval order.")
        return nodes


## Step 6: Run each configuration

In [ ]:
import time

def run_config(label, cfg):
    print("=" * 70)
    print(f"CONFIG {label}: top_k={cfg['top_k']}, threshold={cfg['threshold']}, reranker={cfg['reranker']}")
    print("=" * 70)

    t0 = time.time()
    retriever = index.as_retriever(similarity_top_k=cfg["top_k"])
    retrieved_nodes = retriever.retrieve(query)

    if cfg["threshold"] is not None:
        retrieved_nodes = [n for n in retrieved_nodes if n.score and n.score > cfg["threshold"]]

    if cfg["reranker"]:
        retrieved_nodes = llm_rerank(retrieved_nodes, query, llm)

    retrieve_time = time.time() - t0

    print(f"Chunks retrieved after filtering/reranking: {len(retrieved_nodes)}")
    for i, n in enumerate(retrieved_nodes):
        score_str = f"{n.score:.3f}" if n.score is not None else "n/a"
        print(f"\nChunk {i+1} (score={score_str}):")
        print(n.get_text()[:300])
        print("-" * 50)

    context_block = "\n\n".join(
        f"[Chunk {i}] {n.get_text()}" for i, n in enumerate(retrieved_nodes)
    )
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not fully contained in the context, say what is missing.\n\n"
        f"Context:\n{context_block}\n\nQuestion: {query}\nAnswer:"
    )
    t1 = time.time()
    answer = llm.complete(prompt)
    answer_time = time.time() - t1

    print(f"\nFinal Answer:\n{answer}")
    print(f"\n(retrieve+filter+rerank: {retrieve_time:.3f}s, generate: {answer_time:.3f}s)")

    return {
        "label": label,
        "config": cfg,
        "num_chunks": len(retrieved_nodes),
        "nodes": [(n.score, n.get_text()) for n in retrieved_nodes],
        "answer": str(answer),
        "retrieve_time": retrieve_time,
        "answer_time": answer_time,
    }

results = {}
for label, cfg in configs.items():
    results[label] = run_config(label, cfg)


CONFIG A: top_k=5, threshold=None, reranker=False
Chunks retrieved after filtering/reranking: 5

Chunk 1 (score=0.337):
Description
Test Method
Result
Autoclave - Pump tubing
121C > 15 min
Conforms
Gamma Irradiation - Inlets
25.0 - 40.0 kGy
Conforms
Flow Rate Test
Per SOP-QC-042
Pass
Pressure Integrity
Max 0.5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed, no damage
Pass
Regu
--------------------------------------------------

Chunk 2 (score=0.283):
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKTA ready
29184612
+2 C to +40 C
Operating Instructions access link:
https://cdn.cytivalifesciences.com/api/public/content/digi-14501-pdf
This letter is provide
--------------------------------------------------

Chunk 3 (score=0.261):
Description 
Limits 
Autoclave
Pump tubing
121°C > 15 min 
Gamma Irradiation 
Inlets 
25.0 – 40.0 kGy 
Regulatory Conformance 
B

## Step 7: Comparison summary

Printed directly from the real run above — not estimated.

In [ ]:
print(f"{'Config':<8}{'Chunks':<9}{'Best Score':<12}{'Retrieve(s)':<14}{'Generate(s)':<12}")
for label, r in results.items():
    best_score = r["nodes"][0][0] if r["nodes"] else None
    best_score_str = f"{best_score:.3f}" if best_score is not None else "n/a"
    print(f"{label:<8}{r['num_chunks']:<9}{best_score_str:<12}{r['retrieve_time']:<14.3f}{r['answer_time']:<12.3f}")


Config  Chunks   Best Score  Retrieve(s)   Generate(s) 
A       5        0.337       0.032         7.205       
B       4        0.337       0.021         6.832       
C       4        0.337       4.517         6.759       


## Notes / Design choices

- **Same index across all 3 configs.** Chunking (`chunk_size=200`, `chunk_overlap=30`) and the MiniLM embedding model are built once and reused, so the only variables changing between A, B, and C are the retrieval parameters (`top_k`, threshold, reranker) — not the underlying embeddings or chunk boundaries.
- **Custom LLM reranker, not LlamaIndex's built-in `LLMRerank`.** LlamaIndex's `LLMRerank` postprocessor expects a specific output format from the LLM that is tuned for OpenAI-style models and can fail to parse with other LLMs. We built a small, transparent reranker instead: ask Gemini to return the chunk indices ordered by relevance, and fall back to the original order if parsing fails. This is a genuine, documented engineering trade-off, not a shortcut.
- **Threshold value.** 0.15 was chosen based on the actual score range MiniLM produces on this document (roughly 0.15–0.35 for relevant chunks in this project, per the earlier embedding-model comparison task), per the task's guidance to check real scores before setting a threshold.
- Full comparison results, ratings, and analysis answers are written up in the companion Google Doc (RAG Configurations Comparison Results) using the real chunks, answers, and timings captured above — nothing in that doc is invented independently of this run.
